# 🏷️ The Price Is Right: Data Pipeline & Curation

This notebook walks through the data preparation pipeline for the price prediction project:
1. Extracting raw products from Amazon Reviews 2023 dataset
2. Filtering and scrubbing noisy product metadata ($0.50 - $999.49 bounds)
3. LLM batch summarization into standardized schema
4. Token distribution analysis and formatting SFT prompts for fine-tuning


In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.data.schema import Item
from pricer.data.loader import ItemLoader
from pricer.data.summarizer import Preprocessor
from pricer.data.prompt_formatter import PromptFormatter
from pricer.config import DEFAULT_LITE_DATASET

load_dotenv(override=True)
if os.environ.get('HF_TOKEN'):
    login(os.environ['HF_TOKEN'], add_to_git_credential=True)


## 1. Load Preprocessed Dataset from Hugging Face Hub


In [ ]:
train, val, test = Item.from_hub(DEFAULT_LITE_DATASET)
print(f'Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test items.')
print('\nSample Item Summary:\n', train[0].summary)
print('Actual Price: $', train[0].price)


## 2. Token Analysis & Cutoff Optimization
We analyze the token length of the product summaries to determine the optimal cutoff.


In [ ]:
formatter = PromptFormatter(cutoff=110)
avg_tok, max_tok, cut_count = formatter.analyze_token_distribution(train + val + test)
print(f'Avg tokens: {avg_tok:.1f} | Max: {max_tok} | Cutoff > 110: {cut_count}')

formatter.prepare_dataset(train, val, test)
print('\nPrepared Prompt:')
print(test[0].prompt)
print('Completion:', test[0].completion)
